# Mission 3: 환자 증상 다중 라벨 분류
## 평가 지표(Macro F1) 구현 및 클래스별 임계값(Threshold) 최적화

- **담당**: 권오현 (평가 지표 구현 및 임계값 최적화)
- **대회 공식 평가 지표**: Macro F1-score (9개 증상 F1 단순 산술 평균)
- **핵심 전략**: 9개 증상의 극심한 클래스 불균형(Class Imbalance)을 극복하기 위해, 기본 0.5 대신 클래스별 최적 임계값을 탐색하여 Macro F1 극대화

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# m3 패키지 import 경로 등록
sys.path.insert(0, str(Path.cwd()))
from m3 import (
    TARGET_SYMPTOMS,
    NUM_CLASSES,
    eval_macro_f1,
    apply_thresholds,
    find_best_thresholds,
    get_threshold_curves,
    save_thresholds_json,
    generate_comparison_markdown,
)

# 한글 폰트 및 마이너스 기호 설정
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

print("m3 패키지 모듈 및 환경 세팅 완료")

## 1. 대회 규정 대상 9개 타겟 증상 표준 확인
대회 규정상 모델 평가 대상은 정확히 아래 9개 증상에 한정.

In [ ]:
print(f"총 타겟 클래스 수: {NUM_CLASSES}개")
for i, sym in enumerate(TARGET_SYMPTOMS):
    print(f"[{i}] {sym}")

## 2. 검증 데이터(Validation Set) 및 모델 예측 확률 로드
KoBERT 모델에서 추출된 예측 확률값(`val_probs.npy`)과 정답 레이블(`val_labels.npy`)을 로드.
*(아직 모델 학습 완료 전인 경우, 현실적인 119 통화 데이터의 클래스 불균형 분포를 가진 시뮬레이션 데이터를 자동 생성하여 테스트)*

In [ ]:
probs_path = Path("val_probs.npy")
labels_path = Path("val_labels.npy")

if probs_path.exists() and labels_path.exists():
    y_probs = np.load(probs_path)
    y_true = np.load(labels_path)
    print(f"실제 검증셋 데이터 로드 성공: {y_probs.shape[0]}건")
else:
    print("실제 모델 가중치 출력 파일이 없어 불균형 검증 시뮬레이션 데이터를 생성")
    np.random.seed(42)
    N = 3640  # Validation 세트 통화 규모
    y_true = np.zeros((N, NUM_CLASSES), dtype=int)
    # 9개 증상별 실제 데이터 발현 비율 반영 (희귀 증상 2% ~ 다빈도 증상 25%)
    prior_ratios = [0.03, 0.18, 0.25, 0.14, 0.09, 0.02, 0.06, 0.04, 0.15]
    for c, r in enumerate(prior_ratios):
        y_true[:, c] = (np.random.rand(N) < r).astype(int)
    
    # 학습된 모델의 현실적 로짓 확률 생성 (양성 평균 0.75, 음성 평균 0.1)
    logits = y_true * 2.2 - (1 - y_true) * 2.5 + np.random.normal(0, 1.2, (N, NUM_CLASSES))
    y_probs = 1 / (1 + np.exp(-logits))
    print(f"시뮬레이션 검증셋 생성 완료: {N}건")

# 각 증상별 양성 건수 확인
fig, ax = plt.subplots(figsize=(10, 4))
counts = y_true.sum(axis=0)
bars = ax.bar(TARGET_SYMPTOMS, counts, color="#4A90E2", edgecolor="black", alpha=0.8)
ax.bar_label(bars, fmt="%d건", padding=3)
ax.set_title("검증 데이터셋의 9개 증상별 양성 발생 건수 (클래스 불균형 현황)", fontsize=12, pad=10)
ax.set_ylabel("건수")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 3. [기준선] 기본 임계값 0.5 적용 시 Macro F1 측정
일반적인 이진 분류 기준치인 `Threshold = 0.5`를 일괄 적용했을 때의 값.

In [ ]:
base_preds = apply_thresholds(y_probs, 0.5)
base_macro_f1, base_class_f1 = eval_macro_f1(y_true, base_preds, return_per_class=True)

print(f"[기준선] 기본 임계값 0.5 적용 시 Macro F1: {base_macro_f1:.4f}")
print("-" * 40)
for sym in TARGET_SYMPTOMS:
    print(f"{sym:<10}: F1 = {base_class_f1[sym]:.4f}")

## 4. [핵심 최적화] 9개 증상별 최적 임계값 탐색 및 F1 곡선 시각화
각 증상별로 0.05 ~ 0.95 구간에서 F1 반응 곡선을 추적하고 최고 F1을 달성하는 최적의 지점을 찾는다.

In [ ]:
# 1. 클래스별 최적 임계값 탐색 수행
best_th, base_f1, opt_f1, base_f1s, opt_f1s = find_best_thresholds(
    y_probs, y_true, step=0.01, min_th=0.05, max_th=0.95
)

# 2. 9개 증상별 F1 반응 곡선 시각화 (3x3 Subplots)
curves = get_threshold_curves(y_probs, y_true, step=0.02)
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for idx, sym in enumerate(TARGET_SYMPTOMS):
    th_range, f1_vals = curves[sym]
    ax = axes[idx]
    ax.plot(th_range, f1_vals, color="#2C3E50", lw=2, label="F1 곡선")
    
    # 기준선(0.5) 표시
    ax.axvline(0.5, color="gray", linestyle=":", label="기준 0.5")
    
    # 최적점 표시
    best_point_th = best_th[idx]
    best_point_f1 = opt_f1s[sym]
    ax.scatter([best_point_th], [best_point_f1], color="red", s=80, zorder=5, label=f"최적 ({best_point_th:.2f})")
    
    ax.set_title(f"{sym} (F1: {base_f1s[sym]:.3f} → {best_point_f1:.3f})", fontsize=11, fontweight="bold")
    ax.set_xlabel("Threshold")
    ax.set_ylabel("F1-score")
    ax.set_ylim([0.0, 1.0])
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend(loc="lower left", fontsize=8)

plt.suptitle("9개 증상별 임계값(Threshold) 변화에 따른 F1-score 반응 곡선", fontsize=15, y=0.99)
plt.tight_layout()
plt.show()

## 5. [성과 비교] Before vs After 비교 리포트
기준(0.5) 대비 클래스별 최적화 임계값 적용 시의 F1 상승폭을 표와 막대그래프로 확인.

In [ ]:
# 9개 증상별 Before/After 막대그래프 비교
x = np.arange(NUM_CLASSES)
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
rects1 = ax.bar(x - width/2, [base_f1s[s] for s in TARGET_SYMPTOMS], width, label="기준 (0.5)", color="#BDC3C7", edgecolor="black")
rects2 = ax.bar(x + width/2, [opt_f1s[s] for s in TARGET_SYMPTOMS], width, label="최적 임계값 적용", color="#E74C3C", edgecolor="black")

ax.set_ylabel("F1-score", fontsize=11)
ax.set_title(f"증상별 F1-score 개선도 (전체 Macro F1: {base_f1:.4f} ➡️ {opt_f1:.4f}, +{(opt_f1 - base_f1):.4f})", fontsize=13, pad=10)
ax.set_xticks(x)
ax.set_xticklabels(TARGET_SYMPTOMS, fontsize=10)
ax.set_ylim([0, 1.05])
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

# reports/comparison.md 파일 자동 생성
md_report = generate_comparison_markdown(base_f1, opt_f1, base_f1s, opt_f1s, best_th)
print("reports/comparison.md 생성 완료")

## 6. 최적 임계값 JSON 저장 (`inference.py` 연동용)
도출된 최적 임계값을 JSON 파일로 내보내어, 대회 제출 스크립트인 `inference.py`에서 바로 불러와 사용 가능하도록 함.

In [ ]:
saved_path = save_thresholds_json(best_th)
print(f"최적 임계값 저장 완료: {saved_path}")

# 저장된 임계값 확인
import json
with open(saved_path, "r", encoding="utf-8") as f:
    data = json.load(f)
print(json.dumps(data, ensure_ascii=False, indent=2))